# TODO: Improve this notebook

In [1]:
import fastf1 as ff1
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from fastf1 import plotting

plotting.setup_mpl(mpl_timedelta_support=True, color_scheme="fastf1")

%matplotlib widget

In [2]:
CURRENT_YEAR = 2025

In [3]:
def get_all_incidents_data(sessions: list) -> pd.DataFrame:
    """Filter racing incidents and combine occurrences across sessions."""
    target_flags = ["BLACK", "DOUBLE YELLOW", "RED", "YELLOW"]
    all_frames = []

    for session in sessions:
        messages = session.race_control_messages
        mask = messages["Flag"].str.upper().isin(target_flags)
        filtered = messages[mask].copy()
        filtered = filtered.groupby("Lap", as_index=False).count()
        filtered["Year"] = session.event.year
        all_frames.append(filtered)

    return pd.concat(all_frames, ignore_index=True)

In [4]:
race_2021 = ff1.get_session(2021, "Qatar", "R")
# In 2022, the GP did not happen due to the world cup
race_2023 = ff1.get_session(2023, "Qatar", "R")
race_2024 = ff1.get_session(2024, "Qatar", "R")

req         WARNING 	DEFAULT CACHE ENABLED! (1.43 GB) C:\Users\Vitor\AppData\Local\Temp\fastf1
events      WARNING 	Correcting user input 'Qatar' to 'Qatar Grand Prix'
events      WARNING 	Correcting user input 'Qatar' to 'Qatar Grand Prix'


In [5]:
race_2021.load()
race_2023.load()
race_2024.load()

core           INFO 	Loading data for Qatar Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 44 completed the race distance 00:00.037000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '14', '11', '31', '18', '55', '16', '4', '5', '10', '3',

In [6]:
incidents = get_all_incidents_data([race_2021, race_2023, race_2024])
incidents

,Lap,Time,Category,Message,Status,Flag,Scope,Sector,RacingNumber,Year
0,1,1,1,1,0,1,1,1,0,2021
1,33,4,4,4,0,4,4,4,0,2021
2,34,1,1,1,0,1,1,1,0,2021
3,51,3,3,3,0,3,3,3,0,2021
4,52,1,1,1,0,1,1,1,0,2021
5,1,1,1,1,0,1,1,1,0,2023
6,33,1,1,1,0,1,1,1,0,2023
7,41,1,1,1,0,1,1,1,0,2023
8,1,1,1,1,0,1,1,1,0,2024
9,5,1,1,1,0,1,1,1,0,2024


How many race incidents happen on this circuit?

In [7]:
len(incidents) / len(incidents.Year.unique())

4.333333333333333

In [8]:
min_incidents = incidents.groupby("Year").size().min()
min_incidents

np.int64(3)

In [9]:
last_year_incidents = len(incidents[incidents["Year"] == CURRENT_YEAR - 1])
last_year_incidents

5

What is the average temperature on this circuit?

In [10]:
race_2021.weather_data

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,0 days 00:00:35.648000,27.4,69.1,1011.6,False,33.8,155,0.3
1,0 days 00:01:35.635000,27.4,69.2,1011.7,False,33.5,155,0.4
2,0 days 00:02:35.644000,27.3,69.6,1011.7,False,33.4,106,0.4
3,0 days 00:03:35.660000,27.3,69.7,1011.7,False,32.7,119,0.6
4,0 days 00:04:35.660000,27.2,69.5,1011.7,False,32.7,131,0.5
...,...,...,...,...,...,...,...,...
147,0 days 02:27:36.175000,26.0,76.8,1013.1,False,28.8,153,0.4
148,0 days 02:28:36.173000,26.0,76.8,1013.2,False,28.8,114,0.3
149,0 days 02:29:36.193000,26.0,76.8,1013.1,False,28.9,148,0.3
150,0 days 02:30:36.208000,26.0,76.7,1013.1,False,29.0,98,0.4


In [11]:
race_2023.weather_data

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,0 days 00:00:20.937000,32.3,64.0,1009.5,False,38.1,282,1.1
1,0 days 00:01:20.937000,32.3,64.0,1009.5,False,38.5,278,1.2
2,0 days 00:02:20.936000,32.2,65.0,1009.5,False,38.4,287,1.0
3,0 days 00:03:20.935000,32.3,65.0,1009.5,False,38.1,289,1.0
4,0 days 00:04:20.935000,32.2,65.0,1009.5,False,38.1,291,1.2
...,...,...,...,...,...,...,...,...
154,0 days 02:34:21.412000,30.9,76.0,1009.9,False,35.4,268,0.8
155,0 days 02:35:21.412000,30.9,76.0,1009.7,False,35.4,343,0.6
156,0 days 02:36:21.411000,30.9,76.0,1009.7,False,35.0,287,1.0
157,0 days 02:37:21.410000,30.8,77.0,1009.7,False,34.8,297,1.0


In [12]:
race_2024.laps.pick_quicklaps().columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')